# 🎬 AI Auto Generate Video — chạy trên Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USER/YOUR_REPO/blob/main/AI_auto_video_colab.ipynb)

Notebook này chạy trọn quy trình của repo [huytranvan2010/AI-auto-generate-video](https://github.com/huytranvan2010/AI-auto-generate-video):

**bài báo / script.json → TTS (OmniVoice) → render template (HyperFrames + Chrome) → ghép FFmpeg → `video.mp4` 9:16**

| Bước | Việc làm |
|---|---|
| 1 | Cấu hình, mount Drive, kiểm tra GPU |
| 2 | Cài Node 22, FFmpeg, Chrome, font |
| 3 | Clone repo + `npm install` |
| 4 | Cài OmniVoice |
| 5 | Dựng TTS server (`POST /tts` → mp3) chạy nền ở port 8123 |
| 6 | Tạo `.env.local` + thử giọng |
| 7 | Tạo `script.json` (mẫu / dán tay / nhờ Claude viết từ bài báo) |
| 8 | Chạy pipeline |
| 9 | Xem và tải kết quả |

**Trước khi chạy:** `Runtime → Change runtime type → T4 GPU`, rồi chạy lần lượt từng cell từ trên xuống.

In [ ]:
#@title 1) Cấu hình { display-mode: "form" }
REPO_URL = "https://github.com/huytranvan2010/AI-auto-generate-video.git" #@param {type:"string"}
USE_DRIVE = True #@param {type:"boolean"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/ai-auto-video" #@param {type:"string"}
# Giọng mẫu để clone (khuyến nghị): đường dẫn file wav/mp3 trên Drive. Để trống = giọng ngẫu nhiên.
REF_AUDIO = "" #@param {type:"string"}
# Lời của file giọng mẫu (không bắt buộc, để trống thì OmniVoice tự nhận dạng)
REF_TEXT = "" #@param {type:"string"}

import os, subprocess
REPO_DIR = "/content/AI-auto-generate-video"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

if REF_AUDIO and not os.path.exists(REF_AUDIO):
    print(f"⚠️ Không thấy file giọng mẫu: {REF_AUDIO}")
if not REF_AUDIO:
    print("⚠️ Chưa có REF_AUDIO: mỗi scene có thể ra một giọng khác nhau. Nên dùng giọng mẫu để giọng đọc nhất quán.")

gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print(gpu if gpu else "❌ Không có GPU. Vào Runtime → Change runtime type → T4 GPU rồi chạy lại.")

In [ ]:
%%bash
#@title 2) Cài Node 22, FFmpeg, Chrome, font
set -e
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq

if ! node --version 2>/dev/null | grep -q '^v22'; then
  curl -fsSL https://deb.nodesource.com/setup_22.x | bash - >/dev/null
  apt-get install -y -qq nodejs >/dev/null
fi

apt-get install -y -qq ffmpeg fonts-noto fonts-noto-color-emoji fonts-liberation >/dev/null

if ! command -v google-chrome-stable >/dev/null; then
  wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
  apt-get install -y -qq /tmp/chrome.deb >/dev/null
fi

# Colab chạy bằng root nên Chrome cần --no-sandbox
cat > /usr/local/bin/chrome-nosandbox <<'EOF'
#!/bin/bash
exec /usr/bin/google-chrome-stable --no-sandbox --disable-dev-shm-usage "$@"
EOF
chmod +x /usr/local/bin/chrome-nosandbox
fc-cache -f >/dev/null

echo "node   : $(node --version)"
echo "npm    : $(npm --version)"
echo "ffmpeg : $(ffmpeg -version | head -1)"
echo "chrome : $(google-chrome-stable --version)"

In [ ]:
#@title 3) Clone repo + npm install
import os
if os.path.isdir(f"{REPO_DIR}/.git"):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
!npm install --no-audit --no-fund

# Cho HyperFrames/Puppeteer dùng Chrome đã cài (bản có --no-sandbox)
for k in ("PUPPETEER_EXECUTABLE_PATH", "CHROME_PATH", "CHROMIUM_PATH"):
    os.environ[k] = "/usr/local/bin/chrome-nosandbox"
print("✅ Repo sẵn sàng tại", os.getcwd())

In [ ]:
#@title 4) Cài OmniVoice
!pip install -q omnivoice fastapi "uvicorn[standard]" soundfile anthropic trafilatura
print("✅ Xong. Nếu Colab báo 'Restart session' thì restart rồi chạy lại từ cell 1.")

In [ ]:
%%writefile /content/tts_server.py
import io, os, subprocess, threading
import numpy as np
import soundfile as sf
import torch
from fastapi import FastAPI, HTTPException
from fastapi.responses import Response
from pydantic import BaseModel
from omnivoice import OmniVoice

SR = 24000
SEED = 1234
REF_AUDIO = os.environ.get("REF_AUDIO") or None
REF_TEXT = os.environ.get("REF_TEXT") or None

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device.startswith("cuda") else torch.float32
model = OmniVoice.from_pretrained("k2-fsa/OmniVoice", device_map=device, dtype=dtype)

lock = threading.Lock()
app = FastAPI()


class Req(BaseModel):
    text: str


@app.get("/health")
def health():
    return {"ok": True, "device": device, "ref_audio": bool(REF_AUDIO)}


@app.post("/tts")
def tts(req: Req):
    text = req.text.strip()
    if not text:
        raise HTTPException(status_code=400, detail="empty text")

    kwargs = {"text": text}
    if REF_AUDIO:
        kwargs["ref_audio"] = REF_AUDIO
        if REF_TEXT:
            kwargs["ref_text"] = REF_TEXT

    with lock:
        torch.manual_seed(SEED)
        audio = model.generate(**kwargs)

    a = audio[0]
    if hasattr(a, "detach"):
        a = a.detach().cpu().numpy()
    wav = np.squeeze(np.asarray(a, dtype=np.float32))

    buf = io.BytesIO()
    sf.write(buf, wav, SR, format="WAV")
    proc = subprocess.run(
        ["ffmpeg", "-loglevel", "error", "-i", "pipe:0", "-f", "mp3", "-b:a", "192k", "pipe:1"],
        input=buf.getvalue(), capture_output=True,
    )
    if proc.returncode != 0:
        raise HTTPException(status_code=500, detail=proc.stderr.decode(errors="ignore"))
    return Response(content=proc.stdout, media_type="audio/mpeg")

In [ ]:
#@title 5) Chạy TTS server nền (port 8123)
import os, subprocess, time, requests

subprocess.run(["pkill", "-f", "uvicorn tts_server"], check=False)
time.sleep(1)

env = os.environ.copy()
env["REF_AUDIO"] = REF_AUDIO
env["REF_TEXT"] = REF_TEXT

log = open("/content/tts_server.log", "w")
server = subprocess.Popen(
    ["python", "-m", "uvicorn", "tts_server:app", "--host", "127.0.0.1", "--port", "8123"],
    cwd="/content", env=env, stdout=log, stderr=subprocess.STDOUT,
)

print("Đang tải model OmniVoice (lần đầu mất vài phút)...")
t0 = time.time()
ok = False
while time.time() - t0 < 900:
    if server.poll() is not None:
        break
    try:
        if requests.get("http://127.0.0.1:8123/health", timeout=2).ok:
            ok = True
            break
    except Exception:
        pass
    time.sleep(3)

if ok:
    print(f"✅ TTS server sẵn sàng sau {time.time() - t0:.0f}s")
else:
    print("❌ Server chưa chạy được. Log:")
    print(open("/content/tts_server.log").read()[-3000:])

In [ ]:
#@title 6) Tạo .env.local + thử giọng
import os, requests
from IPython.display import Audio, display

with open(f"{REPO_DIR}/.env.local", "w") as f:
    f.write("TTS_PROVIDER=omnivoice\nOMNIVOICE_ENDPOINT=http://127.0.0.1:8123\nTTS_CONCURRENCY=1\n")

r = requests.post(
    "http://127.0.0.1:8123/tts",
    json={"text": "Xin chào, đây là bản thử giọng đọc tiếng Việt."},
    timeout=300,
)
r.raise_for_status()
with open("/content/test.mp3", "wb") as f:
    f.write(r.content)
display(Audio("/content/test.mp3"))

In [ ]:
#@title 7) Tạo script.json
# ── Chọn cách tạo script ──────────────────────────────────────────────
#   "sample" : dùng script mẫu 3 scene (để thử pipeline nhanh)
#   "paste"  : dán script.json của bạn vào SCRIPT_PASTE
#   "claude" : nhờ Claude viết script.json từ bài báo (cần ANTHROPIC_API_KEY)
SCRIPT_MODE = "sample"

# Dùng cho "claude": điền URL hoặc dán nội dung bài (nếu cả hai có thì ưu tiên ARTICLE_TEXT)
ARTICLE_URL = ""
ARTICLE_TEXT = """"""
CLAUDE_MODEL = "claude-sonnet-5-5"

# Dùng cho "paste"
SCRIPT_PASTE = r"""
"""

import json, os, re, time, unicodedata
from urllib.parse import urlparse

SAMPLE = {
    "version": "1.0", "renderer": "hyperframes", "aspect": "9:16",
    "metadata": {"title": "Apple ra mắt iPhone 17 camera 200MP",
                 "source": {"url": "https://example.com", "domain": "example.com", "image": None},
                 "channel": "AI Coding"},
    "voice": {"provider": "omnivoice", "speed": 1.0},
    "scenes": [
        {"id": "hook", "type": "hook",
         "voiceText": "Apple vừa ra mắt iPhone mười bảy với camera hai trăm megapixel.",
         "templateId": "frame-liquid-bg-hero",
         "inputs": {"kicker": "🔥 Tin nóng", "headline": "iPhone 17", "subheadline": "Camera 200MP",
                    "cta": "Theo dõi ngay", "brand": "AI Coding"}},
        {"id": "body-1", "type": "body",
         "voiceText": "Cảm biến mới thu nhiều ánh sáng hơn, ảnh đêm sắc nét hơn rõ rệt.",
         "templateId": "frame-pentagram-stat",
         "inputs": {"label": "Camera", "headline": "200MP",
                    "subtitle": "Cảm biến lớn nhất từ trước tới nay", "anchor": "200"}},
        {"id": "outro", "type": "outro",
         "voiceText": "Theo dõi AI Coding để xem bản tin công nghệ mới mỗi ngày.",
         "templateId": "frame-logo-outro",
         "inputs": {"brand_name": "AI Coding", "tagline": "Tin công nghệ mỗi ngày",
                    "primary_url": "https://aicodingvn.vercel.app/"}},
    ],
}

EMOJI_RE = re.compile("[\U0001F000-\U0001FAFF\u2600-\u27BF]")

def slugify(s, n=40):
    s = s.replace("đ", "d").replace("Đ", "D")
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    s = re.sub(r"[^a-zA-Z0-9]+", "-", s).strip("-").lower()
    return (s[:n].strip("-")) or "video"

def validate(d):
    errs = []
    if d.get("version") != "1.0": errs.append('version phải là "1.0"')
    if d.get("renderer") != "hyperframes": errs.append('renderer phải là "hyperframes"')
    sc = d.get("scenes", [])
    if not 3 <= len(sc) <= 12: errs.append(f"cần 3–12 scene, hiện có {len(sc)}")
    if sc and sc[0].get("type") != "hook": errs.append("scene đầu phải là hook")
    if sc and sc[-1].get("type") != "outro": errs.append("scene cuối phải là outro")
    for s in sc:
        if not os.path.isdir(f"{REPO_DIR}/templates/{s.get('templateId')}"):
            errs.append(f"templateId không tồn tại: {s.get('templateId')}")
        if EMOJI_RE.search(s.get("voiceText", "")):
            errs.append(f"voiceText scene {s.get('id')} có emoji")
    return errs

def fetch_article(url):
    import trafilatura
    html = trafilatura.fetch_url(url)
    if not html:
        raise RuntimeError("Không tải được URL (paywall/JS?). Hãy dán nội dung vào ARTICLE_TEXT.")
    text = trafilatura.extract(html) or ""
    meta = trafilatura.extract_metadata(html)
    return (meta.title if meta and meta.title else url), text

def generate_with_claude():
    from anthropic import Anthropic
    try:
        from google.colab import userdata
        key = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        import getpass
        key = getpass.getpass("ANTHROPIC_API_KEY: ")

    if ARTICLE_TEXT.strip():
        title, content = ARTICLE_TEXT.strip().split("\n", 1)[0][:80], ARTICLE_TEXT.strip()
    elif ARTICLE_URL.strip():
        title, content = fetch_article(ARTICLE_URL.strip())
    else:
        raise RuntimeError("Hãy điền ARTICLE_URL hoặc ARTICLE_TEXT.")

    skill = open(f"{REPO_DIR}/.claude/skills/create-template-video/SKILL.md", encoding="utf-8").read()
    catalog = open(f"{REPO_DIR}/templates/CATALOG.md", encoding="utf-8").read()
    domain = urlparse(ARTICLE_URL).netloc if ARTICLE_URL.strip() else "local"

    system = (
        "Bạn thực hiện Step 5 và Step 6 của skill dưới đây để viết script.json cho video. "
        "Chỉ trả về MỘT đối tượng JSON hợp lệ (không markdown, không giải thích).\n\n"
        "=== SKILL ===\n" + skill + "\n\n=== CATALOG TEMPLATE ===\n" + catalog
    )
    user = (f"Tiêu đề: {title}\nURL: {ARTICLE_URL or 'local'}\nDomain: {domain}\n"
            f"ogImage: null\n\nNội dung bài:\n{content[:12000]}")

    client = Anthropic(api_key=key)
    msgs = [{"role": "user", "content": user}]
    for attempt in range(3):
        resp = client.messages.create(model=CLAUDE_MODEL, max_tokens=12000, system=system, messages=msgs)
        text = resp.content[0].text
        try:
            data = json.loads(text[text.index("{"): text.rindex("}") + 1])
            errs = validate(data)
        except Exception as e:
            data, errs = None, [f"JSON không hợp lệ: {e}"]
        if not errs:
            return data
        print(f"Lần {attempt + 1} lỗi: {errs}")
        msgs += [{"role": "assistant", "content": text},
                 {"role": "user", "content": "Lỗi: " + "; ".join(errs) + ". Sửa lại và chỉ trả JSON."}]
    raise RuntimeError("Claude chưa tạo được script hợp lệ sau 3 lần.")

if SCRIPT_MODE == "sample":
    script = SAMPLE
elif SCRIPT_MODE == "paste":
    script = json.loads(SCRIPT_PASTE)
elif SCRIPT_MODE == "claude":
    script = generate_with_claude()
else:
    raise ValueError("SCRIPT_MODE phải là sample / paste / claude")

errs = validate(script)
if errs:
    raise ValueError("script.json chưa hợp lệ: " + "; ".join(errs))

slug = slugify(script["metadata"]["title"])
out_dir = f"output/{slug}-{time.strftime('%Y%m%d-%H%M')}"
os.makedirs(f"{REPO_DIR}/{out_dir}", exist_ok=True)
SCRIPT_PATH = f"{out_dir}/script.json"
with open(f"{REPO_DIR}/{SCRIPT_PATH}", "w", encoding="utf-8") as f:
    json.dump(script, f, ensure_ascii=False, indent=2)

print("✅ Đã ghi", SCRIPT_PATH)
for s in script["scenes"]:
    print(f"  [{s['type']:5}] {s['id']:10} {s['templateId']}")

In [ ]:
#@title 8) Chạy pipeline (TTS → render → ghép video)
import os
os.chdir(REPO_DIR)
!npm run pipeline -- {SCRIPT_PATH}

In [ ]:
#@title 9) Xem, tải và lưu kết quả
import os, shutil, subprocess
from IPython.display import Video, display
from google.colab import files

out = f"{REPO_DIR}/{os.path.dirname(SCRIPT_PATH)}"
video = f"{out}/video.mp4"
assert os.path.exists(video), "Chưa có video.mp4, kiểm tra log ở cell 8."

dur = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                      "-of", "default=nw=1:nk=1", video], capture_output=True, text=True).stdout.strip()
size_mb = os.path.getsize(video) / 1e6
print(f"🎬 {video}\n   {float(dur):.1f}s · {size_mb:.1f} MB")

if size_mb < 40:
    display(Video(video, embed=True, width=270))

if USE_DRIVE:
    dest = f"{DRIVE_OUTPUT_DIR}/{os.path.basename(out)}"
    shutil.copytree(out, dest, dirs_exist_ok=True)
    print("💾 Đã lưu vào Drive:", dest)

for name in ("video.mp4", "voice.mp3", "script.txt"):
    p = f"{out}/{name}"
    if os.path.exists(p):
        files.download(p)

## 🛠️ Xử lý sự cố

| Triệu chứng | Cách xử lý |
|---|---|
| Cell 5 báo server lỗi | Xem log cuối cell, thường do thiếu GPU hoặc `pip` chưa cài xong (restart session rồi chạy lại từ cell 1) |
| `hyperframes render failed` / Chrome không mở được | Chạy lại cell 2 và 3 (đặt đường dẫn Chrome), đảm bảo cell 2 in ra phiên bản Chrome |
| Chữ tiếng Việt hoặc emoji bị ô vuông | Chạy lại cell 2 (cài `fonts-noto`, `fonts-noto-color-emoji`) |
| Mỗi scene một giọng khác nhau | Điền `REF_AUDIO` (file giọng mẫu trên Drive) ở cell 1 rồi chạy lại cell 5 |
| Muốn đọc lại / render lại 1 scene | Xoá `voice/scene-<id>.mp3` hoặc `clips/scene-<id>.mp4` trong thư mục output rồi chạy lại cell 8 |
| Số bị đọc sai | Viết số ra chữ trong `voiceText` (xem `.claude/skills/create-template-video/SKILL.md`) |
| Runtime bị ngắt | Output đã nằm trên Drive (`DRIVE_OUTPUT_DIR`), chạy lại cell 1 → 6 |

**Hiệu ứng âm thanh (SFX):** không bắt buộc. Pipeline vẫn ra video bình thường, chỉ là không có SFX.

**Dùng Claude viết script:** thêm khoá `ANTHROPIC_API_KEY` vào Colab Secrets (biểu tượng chìa khoá bên trái), bật *Notebook access*, rồi đặt `SCRIPT_MODE = "claude"` ở cell 7.